# MedSAM como refinador da DR-IFT

Esta é uma **ablação controlada na validação congelada**. Repetiremos DR-IFT e os três modos do SAM 1 e acrescentaremos somente o `MedSAM_box`. Assim, isolamos o efeito do pré-treinamento médico.

A arquitetura central continua sendo a **DR-IFT**: ela produz a probabilidade e todas as caixas. O MedSAM não vê a referência. Teste interno e FUSeg permanecem fechados.


## Hipótese e regra definida antes do resultado

O MedSAM pode interpretar melhor imagens médicas que o SAM genérico, mas suas imagens de treinamento são muito diferentes de fotografias de úlceras do pé. Por isso ele é uma hipótese, não uma melhoria presumida.

Ele só avançará se, contra a DR-IFT, obtiver simultaneamente: ganho médio de Dice ≥ **0,01** e limite inferior do IC 95% pareado > **0**. Esse 0,01 é um critério operacional do estudo, não um limiar clínico universal.


## 1. Preparar o Colab

O Dataset é copiado ao disco temporário para acelerar a leitura; projeto e resultados continuam no Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys, urllib.request, yaml
PROJECT_ROOT=Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
DATA_ROOT=Path('/content/tcc_data')
%cd {PROJECT_ROOT}
assert PROJECT_ROOT.is_dir(), f'Projeto ausente: {PROJECT_ROOT}'
source=PROJECT_ROOT/'Dataset'
assert source.is_dir(), f'Dataset ausente: {source}'
shutil.copytree(source,DATA_ROOT/'Dataset',dirs_exist_ok=True)
assert (DATA_ROOT/'Dataset/DFUC2022_train_images/100001.jpg').is_file()
gpu=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
print('Cache local pronto | GPU:',gpu)


## 2. Instalar a implementação congelada

SAM e MedSAM usam a mesma arquitetura ViT-B do pacote oficial do SAM. O código fica fixado no commit já auditado. `scikit-image` é necessário para reproduzir o redimensionamento oficial do MedSAM.


In [ ]:
SAM_COMMIT = "6fdee8f2727f4506cfbbe553e23b895e27956588"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/facebookresearch/segment-anything.git@{SAM_COMMIT}",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyyaml",
        "pytest",
        "matplotlib",
        "scikit-image",
    ],
    check=True,
)
print("Dependências prontas")


## 3. Obter e verificar os dois checkpoints

Os pesos ficam em cache no Drive. Mesmo se já existirem, calculamos SHA-256: um arquivo incompleto ou diferente interrompe a execução.


In [ ]:
def download_if_missing(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.is_file():
        print("Baixando", path.name, "...")
        urllib.request.urlretrieve(url, path)


def sha(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


sam = PROJECT_ROOT / "models/sam/sam_vit_b_01ec64.pth"
medsam = PROJECT_ROOT / "models/medsam/medsam_vit_b.pth"
download_if_missing("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", sam)
download_if_missing("https://zenodo.org/records/10689643/files/medsam_vit_b.pth?download=1", medsam)
expected = {
    "sam_vit_b_01ec64.pth": "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912",
    "medsam_vit_b.pth": "34b34b78c1d18cb8c6bf84cf9c00e135d6d6c965699f3c0e31ef1bc9dcb5be74",
}
for path in (sam, medsam):
    actual = sha(path)
    assert actual == expected[path.name], f"Hash inválido: {path}"
    print(path.name, f"{path.stat().st_size/1024**2:.1f} MiB", "| hash aprovado")


## 4. Auditar o protocolo

Aqui conferimos DR-IFT seed 42/época 56/threshold 0,18, as 294 imagens e os manifests congelados. Nenhum threshold será procurado novamente.


In [ ]:
CONFIG = PROJECT_ROOT / "configs/medsam_validation.yaml"
cfg = yaml.safe_load(CONFIG.read_text())
manifest = PROJECT_ROOT / cfg["data"]["manifest"]
fixed = PROJECT_ROOT / cfg["data"]["fixed_visualization_manifest"]
with manifest.open(encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 294 and all(r["split"] == "validation" for r in rows)
assert sha(manifest) == cfg["data"]["manifest_sha256"]
assert sha(fixed) == cfg["data"]["fixed_visualization_manifest_sha256"]
assert cfg["network"]["threshold"] == 0.18 and cfg["medsam"]["prompt_mode"] == "box"
print(
    "DR-IFT:",
    cfg["network"]["run_id"],
    "| época",
    cfg["network"]["checkpoint_epoch"],
    "| threshold",
    cfg["network"]["threshold"],
)
print("Comparação:", cfg["evaluation"]["variants"])
print("Validação congelada:", len(rows), "| teste/FUSeg: fechados")


## 5. Executar os testes

Os testes impedem mudança acidental das caixas, variantes, estatística ou partições.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_sam_refinement.py",
        "tests/test_sam_validation.py",
        "tests/test_medsam_validation.py",
    ],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 6. Executar a ablação

Cada imagem passa uma vez pela DR-IFT. Depois, as mesmas caixas alimentam SAM e MedSAM. O MedSAM usa o fluxo oficial: quadrado 1024, min–max, box em coordenadas 1024 e saída >0,5. Aguarde `status: completed`.


In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"\nreturncode={code}")
    if code:
        raise subprocess.CalledProcessError(code, command)


run_stream(
    [
        sys.executable,
        "scripts/run_medsam_validation.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--config",
        "configs/medsam_validation.yaml",
    ]
)


## 7. Ler números e decisão

O desempenho absoluto não basta: a tabela pareada mede a mudança na **mesma imagem**. O IC 95% mostra a incerteza e Holm controla os quatro contrastes SAM-versus-DR-IFT.


In [ ]:
import pandas as pd
from IPython.display import display

runs = sorted(
    (PROJECT_ROOT / "results/day13b/MedSAM_validation").glob(
        "MedSAM_ViTB_DRIFT_seed42_validation_*"
    )
)
assert runs, "Execução ausente"
RUN = runs[-1]
summary = pd.read_csv(RUN / "variant_summary.csv")
effects = pd.read_csv(RUN / "paired_comparisons.csv")
decision = json.loads((RUN / "advance_decision.json").read_text())
display(
    summary[
        [
            "variant",
            "images",
            "dice_mean",
            "iou_mean",
            "precision_mean",
            "recall_mean",
            "boundary_iou_mean",
            "zero_dice_images",
            "mean_postprocess_seconds",
        ]
    ]
)
display(
    effects[(effects.metric == "dice")][
        [
            "comparison",
            "mean_difference",
            "ci95_low",
            "ci95_high",
            "improved",
            "equal",
            "worsened",
            "permutation_p_holm",
        ]
    ]
)
print(json.dumps(decision, indent=2, ensure_ascii=False))


## 8. Ver os 12 casos congelados

O painel completo permite comparar todos os métodos. No mapa de erros: verde = interseção, laranja = falso positivo e vermelho = falso negativo.


In [ ]:
from PIL import Image

display(Image.open(RUN / "medsam_validation_summary.png"))
display(Image.open(RUN / "fixed_validation_panel_all_models.png"))
display(Image.open(RUN / "fixed_validation_panel_medsam.png"))


## 9. Empacotar para auditoria

O ZIP contém protocolo, métricas por imagem, incerteza, auditoria dos prompts e painéis; não duplica Dataset ou checkpoints.


In [ ]:
export = PROJECT_ROOT / "exports/day13b"
export.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(export / f"medsam_validation_{RUN.name}"), "zip", RUN)
print("Arquivo exportado:", archive)


## Interpretação

A análise avalia se o MedSAM, condicionado à saída da DR-IFT, atende ao critério pré-registrado para avançar às avaliações finais.
